# Tomato Vision: jalankan model TensorFlow asli

Notebook ini menjalankan model **TensorFlow/Keras** yang sama dengan yang diekspor ke demo web, pada foto tomat Anda.
Foto yang diunggah hanya ada di sesi Colab ini.

> **Status:** notebook ini sudah dijalankan di Python lokal (jalur tanpa Colab), tetapi **belum diuji di Colab**.
> Repo dipin ke Python 3.11 / TensorFlow 2.16.1 (`requirements.txt`); runtime Colab bisa berbeda. Bila instalasi gagal, lihat sel 2.

Hasil adalah dugaan kondisi visual satu tomat, bukan penilaian keamanan pangan.

## 1. Ambil kode

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/sellebeww/tomato-vision.git"
# Di dalam repo (mis. dijalankan lokal) tidak perlu clone; di Colab repo di-clone ke /content.
if not Path("src/predict.py").exists() and Path("../src/predict.py").exists():
    os.chdir("..")                            # dijalankan dari folder notebooks/ di dalam repo
if not Path("src/predict.py").exists():
    if not Path("tomato-vision").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    os.chdir("tomato-vision")
print("Folder kerja:", os.getcwd())
print("Python:", sys.version.split()[0])

## 2. Pasang dependensi

CPU saja sudah cukup. Di Colab langkah ini mengunduh TensorFlow 2.16.1 (beberapa menit) dan dilewati bila versi yang dipin sudah terpasang. Versi yang dipin tersedia untuk Python 3.9–3.12; bila gagal (mis. runtime Python 3.13), notebook memakai TensorFlow bawaan runtime dengan peringatan.

In [ ]:
import importlib.metadata as md

def version(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None

PINNED = {"tensorflow": "2.16.1", "keras": "3.15.1"}
if all(version(p) == v for p, v in PINNED.items()):
    print("TensorFlow 2.16.1 / Keras 3.15.1 sudah terpasang, instalasi dilewati.")
else:
    if not sys.version.startswith(("3.11", "3.12")):
        print("PERINGATAN: versi yang dipin hanya tersedia untuk Python 3.9-3.12; runtime ini Python", sys.version.split()[0])
    install = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], capture_output=True, text=True)
    if install.returncode == 0:
        print("Dependensi terpasang. Bila Colab meminta, pilih Runtime > Restart session lalu jalankan ulang dari sel 1.")
    elif version("tensorflow") and version("keras") and version("keras").startswith("3."):
        print(install.stderr[-800:])
        print(f"PERINGATAN: versi yang dipin gagal dipasang; memakai TensorFlow {version('tensorflow')} / Keras {version('keras')} bawaan runtime. "
              "Kombinasi ini belum diuji dan hasilnya bisa sedikit berbeda dari demo web.")
    else:
        raise RuntimeError("Instalasi gagal dan tidak ada TensorFlow/Keras 3 yang bisa dipakai:\n" + install.stderr[-2000:])

## 3. Siapkan foto

Unggah satu atau beberapa foto (JPG/PNG/WebP). Tanpa unggahan, dipakai foto contoh dari repo.

In [ ]:
import shutil
PHOTOS = Path("uploads")
shutil.rmtree(PHOTOS, ignore_errors=True)
PHOTOS.mkdir()
try:
    from google.colab import files           # hanya ada di Colab
    uploaded = files.upload()
    for name, data in uploaded.items():
        (PHOTOS / Path(name).name).write_bytes(data)
except ImportError:
    print("Bukan di Colab: unggahan dilewati.")
if not any(PHOTOS.iterdir()):
    for example in sorted(Path("site/examples").glob("*.jpg")):
        shutil.copy(example, PHOTOS / example.name)
    print("Memakai foto contoh dari repo.")
print(sorted(p.name for p in PHOTOS.iterdir()))

## 4. Prediksi

Perintah yang sama dengan penggunaan lokal: `python -m src.predict uploads/`. Preprocessing dan ambang "perlu tinjauan" identik dengan demo web.

In [ ]:
result = subprocess.run([sys.executable, "-m", "src.predict", str(PHOTOS), "--json"], capture_output=True, text=True)
if result.returncode not in (0, 1):
    raise RuntimeError(result.stderr[-2000:])
import json
rows = json.loads(result.stdout)
print(len(rows), "foto diproses")

## 5. Probabilitas per kelas

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

classes = ["segar", "tidak_segar", "busuk"]
for row in rows:
    fig, (left, right) = plt.subplots(1, 2, figsize=(8, 2.6), gridspec_kw={"width_ratios": [1, 2]})
    if "error" in row:
        left.axis("off"); right.axis("off")
        fig.suptitle(Path(row["image"]).name + "\nGAGAL: " + row["error"], fontsize=9)
    else:
        with Image.open(row["image"]) as im:
            left.imshow(ImageOps.exif_transpose(im).convert("RGB"))
        left.axis("off")
        values = [row["probabilities"][c] for c in classes]
        right.barh(classes[::-1], values[::-1], color="#1c5339")
        right.set_xlim(0, 1)
        for y, v in enumerate(values[::-1]):
            right.text(min(v + 0.02, 0.8), y, f"{v:.1%}", va="center")
        status = "PERLU TINJAUAN" if row["needs_review"] else "kandidat (bukan jaminan)"
        fig.suptitle(f"{Path(row['image']).name}: {row['label']} ({row['confidence']:.1%}), {status}", fontsize=9)
        if row["review_reasons"]:
            print(Path(row["image"]).name, "->", "; ".join(row["review_reasons"]))
    plt.tight_layout(); plt.show()